# Actividad: Procesamiento Distribuido con Apache Spark
Este notebook contiene el procesamiento, limpieza y análisis de un dataset de e-commerce a gran escala.

In [1]:
!pip install pyspark -q

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as fct

# 1. Configuración de la sesión de Spark optimizada
spark = SparkSession.builder \
    .appName("EcommerceBigDataAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Sesión de Spark iniciada correctamente.")

Sesión de Spark iniciada correctamente.


In [11]:
# Información sobre los primeros 5 registros del archivo de datos
!head -n 5 big_data_ecommerce_logs.csv

ID_Sesion,Timestamp,ID_Usuario,Evento,Monto,Latencia_ms,Dispositivo,Ciudad
SES-919516,2024-01-11 15:25:16,14508.0,add_to_cart,973.28,167,Android,Bogotá
SES-448787,2024-01-06 04:39:47,11371.0,view,872.86,499,Android,Barranquilla
SES-404985,2024-01-05 16:29:45,18942.0,click,448.69,312,Android,Bogotá
SES-613097,2024-01-08 02:18:17,45193.0,click,430.88,246,Android,Bogotá


In [8]:
# 2. Carga del dataset
file_path = '/content/big_data_ecommerce_logs.csv'
data_ecommerce = spark.read.csv(file_path, header=True, inferSchema=True)

print(f"Dataset cargado. Total de registros: {data_ecommerce.count():,}")
data_ecommerce.printSchema()

Dataset cargado. Total de registros: 1,020,000
root
 |-- ID_Sesion: string (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- ID_Usuario: double (nullable = true)
 |-- Evento: string (nullable = true)
 |-- Monto: double (nullable = true)
 |-- Latencia_ms: integer (nullable = true)
 |-- Dispositivo: string (nullable = true)
 |-- Ciudad: string (nullable = true)



## Calidad de Datos a Gran Escala



### Pregunta 1: Integridad de Sesiones
**¿Cuántos registros duplicados existen en el dataset y cuál es la estrategia de Spark más eficiente para eliminarlos sin saturar la memoria?**

La estrategia más eficiente es utilizar `dropDuplicates()`. Spark maneja esto mediante un proceso distribuido de hashing y shuffling, lo que permite identificar duplicados a través de los ejecutores sin necesidad de cargar todo el dataset en el driver.

In [28]:
total_count = data_ecommerce.count()

# Aplicamos la estrategia eficiente de dropDuplicates
df_unique = data_ecommerce.dropDuplicates()
unique_count = df_unique.count()
duplicates_count = total_count - unique_count

print(f"Registros totales: {total_count:,}")
print(f"Registros duplicados detectados: {duplicates_count:,}")
print(f"Registros tras limpieza: {unique_count:,}")

Registros totales: 1,020,000
Registros duplicados detectados: 18,832
Registros tras limpieza: 1,001,168


### Pregunta 2: Normalización de Eventos
**¿Cómo se pueden unificar categorías redundantes como 'view' y 'VIEW' y cuál es el impacto en el conteo total?**

Utilizamos la función nativa `lower()` para estandarizar la columna, lo que permite que Spark agrupe correctamente los eventos que solo difieren en el uso de mayúsculas. Al convertir todos los eventos a minúsculas, hemos unificado las categorías redundantes. Los resultados muestran que el evento más común es view con 431,531 registros

In [31]:
from pyspark.sql.functions import lower, col

# Normalización de la columna Evento a minúsculas
df_normalized = df_unique.withColumn("Evento", lower(col("Evento")))

# Conteo de eventos antes de la corrección
event_counts_antes = df_unique.groupBy("Evento").count().orderBy("count", ascending=False)
print("Distribución de eventos antes la unificación:")
event_counts_antes.show()

# Conteo de eventos corregidos
event_counts = df_normalized.groupBy("Evento").count().orderBy("count", ascending=False)

print("Distribución de eventos tras la unificación:")
event_counts.show()

Distribución de eventos antes la unificación:
+-----------+------+
|     Evento| count|
+-----------+------+
|       view|401572|
|      click|300169|
|add_to_cart|199439|
|   purchase| 49807|
|       VIEW| 29959|
|click_error| 20222|
+-----------+------+

Distribución de eventos tras la unificación:
+-----------+------+
|     Evento| count|
+-----------+------+
|       view|431531|
|      click|300169|
|add_to_cart|199439|
|   purchase| 49807|
|click_error| 20222|
+-----------+------+



### Pregunta 3: Gestión de Nulos en Dispositivos
**¿Cuántos valores nulos hay en la columna 'Dispositivo' y qué impacto porcentual tendría su eliminación sobre el dataset actual?**

Detectamos 50,056 valores nulos en la columna 'Dispositivo'. Si decidimos eliminarlos, la pérdida de datos es exactamente del 5.00%, lo cual es un margen aceptable para este tipo de análisis a gran escala. Tras la limpieza, el dataset cuenta con 951,112 registros

In [32]:
from pyspark.sql.functions import isnull

# 1. Conteo inicial antes de la limpieza de nulos en Dispositivo
count_before = df_normalized.count()
null_dispositivo_count = df_normalized.filter(isnull(col("Dispositivo"))).count()

# 2. Cálculo del impacto porcentual
porcentaje_perdida = (null_dispositivo_count / count_before) * 100

# 3. Limpieza y conteo final
df_no_null_dispositivo = df_normalized.dropna(subset=["Dispositivo"])
count_after = df_no_null_dispositivo.count()

print(f"Registros antes de limpiar nulos en Dispositivo: {count_before:,}")
print(f"Cantidad de valores nulos detectados en Dispositivo: {null_dispositivo_count:,}")
print(f"Porcentaje de pérdida de datos: {porcentaje_perdida:.2f}%")
print(f"Registros después de la limpieza: {count_after:,}")

Registros antes de limpiar nulos en Dispositivo: 1,001,168
Cantidad de valores nulos detectados en Dispositivo: 50,056
Porcentaje de pérdida de datos: 5.00%
Registros después de la limpieza: 951,112


## Análisis de Rendimiento y Outliers

### Pregunta 4: Detección de Latencia Crítica
**Identifica los registros donde la Latencia_ms supera los 10,000 ms. ¿Cuántos registros presentan el valor extremo de 999,999 ms y qué impacto tienen en el promedio global de latencia del sistema?**

Se detectaron **482 registros** con latencia superior a 10,000 ms, y todos ellos presentan el valor extremo de **999,999 ms**. Este pequeño grupo de datos está distorsionando drásticamente el promedio global, aumentando el promedio global en **506.65 ms** (pasando de 254.76 ms a 761.41 ms). Es imperativo tratar estos valores como errores de medición o fallos críticos para no sesgar el análisis de experiencia de usuario.

In [33]:
from pyspark.sql.functions import avg, count, when

# 1. Identificar latencias críticas (> 10,000 ms)
critical_latency_df = df_no_null_dispositivo.filter(col("Latencia_ms") > 10000)
critical_count = critical_latency_df.count()

# 2. Identificar outliers específicos (999,999 ms)
outlier_count = df_no_null_dispositivo.filter(col("Latencia_ms") == 999999).count()

# 3. Calcular impacto en el promedio
avg_with_outliers = df_no_null_dispositivo.select(avg("Latencia_ms")).first()[0]
avg_without_outliers = df_no_null_dispositivo.filter(col("Latencia_ms") < 999999).select(avg("Latencia_ms")).first()[0]

print(f"Registros con latencia > 10,000ms: {critical_count:,}")
print(f"Registros con valor extremo (999,999ms): {outlier_count:,}")
print(f"Promedio global con outliers: {avg_with_outliers:.2f} ms")
print(f"Promedio global sin outliers: {avg_without_outliers:.2f} ms")
print(f"Impacto del outlier en el promedio: {avg_with_outliers - avg_without_outliers:.2f} ms")

Registros con latencia > 10,000ms: 482
Registros con valor extremo (999,999ms): 482
Promedio global con outliers: 761.41 ms
Promedio global sin outliers: 254.76 ms
Impacto del outlier en el promedio: 506.65 ms


### Pregunta 5: Análisis de Compras Extremas

**Filtra las transacciones con un Monto de $5,000,000. Compara la media aritmética frente a la mediana del monto total de ventas para demostrar la sensibilidad de la media ante estos errores.**

Identificamos **91 registros** con el monto extremo de \$5,000,000.

La comparación revela un sesgo masivo: la media (\$990.68) es casi el doble de la mediana (\$502.47). Este resultado evidencia que unos pocos valores erróneos o atípicos pueden invalidar conclusiones basadas únicamente en promedios simples, resaltando la necesidad de limpieza de datos o el uso de estadísticos robustos.

In [34]:
from pyspark.sql.functions import median, expr

# 1. Identificar registros con montos extremos
extreme_monto_count = df_no_null_dispositivo.filter(col("Monto") == 5000000).count()

# 2. Calcular Media vs Mediana del Monto
stats = df_no_null_dispositivo.select(
    avg("Monto").alias("Media"),
    expr("percentile_approx(Monto, 0.5)").alias("Mediana")
).first()

media_monto = stats["Media"]
mediana_monto = stats["Mediana"]

print(f"Registros con Monto de $5,000,000: {extreme_monto_count:,}")
print(f"Media aritmética del Monto: ${media_monto:,.2f}")
print(f"Mediana del Monto: ${mediana_monto:,.2f}")
print(f"Diferencia (Sesgo): ${media_monto - mediana_monto:,.2f}")

Registros con Monto de $5,000,000: 91
Media aritmética del Monto: $990.68
Mediana del Monto: $502.47
Diferencia (Sesgo): $488.21


### Pregunta 6: Validación Lógica

**Encuentra registros donde el evento sea 'view' pero el Monto sea mayor a cero. ¿Cuántos registros presentan esta inconsistencia y qué acción de limpieza recomiendas?**

Se encuentran 401,672 registros marcados como 'view' cuyos montos asociados son superiores a cero. Como una visualización no es una transacción económica, la mejor estrategia es la imputación de datos. Se aplica una corrección para establecer estos montos en 0.0, asegurando la integridad lógica del dataset

In [42]:
# 1. Identificar inconsistencias (view con monto > 0)
inconsistent_view_df = df_no_null_dispositivo.filter((col("Evento") == "view") & (col("Monto") > 0))
inconsistent_count = inconsistent_view_df.count()

# 2. Mostrar una muestra de la inconsistencia
print(f"Registros con inconsistencia ('view' con Monto > 0): {inconsistent_count:,}")
if inconsistent_count > 0:
    inconsistent_view_df.select("ID_Sesion", "Evento", "Monto").show(5)

# 3. Recomendación de limpieza:
# Se recomienda imputar el valor de Monto a 0.0 para estos registros,
# ya que el evento 'view' no debería representar una transacción económica.
df_final = df_no_null_dispositivo.withColumn(
    "Monto",
    when((col("Evento") == "view") & (col("Monto") > 0), 0.0).otherwise(col("Monto"))
)

print("\nLimpieza realizada: Montos en eventos 'view' han sido corregidos a 0.0.")
print(df_final.select("ID_Sesion", "Evento", "Monto").where(df_final.Evento == "view").show(5))

Registros con inconsistencia ('view' con Monto > 0): 401,672
+----------+------+------+
| ID_Sesion|Evento| Monto|
+----------+------+------+
|SES-409981|  view|637.57|
| SES-48054|  view|439.96|
|SES-202653|  view|725.98|
|SES-124522|  view|723.88|
|SES-769649|  view|211.38|
+----------+------+------+
only showing top 5 rows

Limpieza realizada: Montos en eventos 'view' han sido corregidos a 0.0.
+----------+------+-----+
| ID_Sesion|Evento|Monto|
+----------+------+-----+
|SES-409981|  view|  0.0|
| SES-48054|  view|  0.0|
|SES-202653|  view|  0.0|
|SES-124522|  view|  0.0|
|SES-769649|  view|  0.0|
+----------+------+-----+
only showing top 5 rows
None


## Insights de Negocio y Agregaciones

### Pregunta 7: Conversión por Dispositivo
**¿Cuál es el dispositivo que registra el mayor número de eventos tipo 'purchase'? Presenta un ranking de mayor a menor.**

El ranking muestra que Android es el líder indiscutible con 19,966 compras, seguido por iOS con 15,069. La plataforma Web ocupa el tercer lugar (9,841) y las Tablets el último (2,442). Estos datos sugieren que la mayoría de las transacciones exitosas provienen de ecosistemas móviles Android.

In [43]:
# 1. Filtrar solo eventos de compra
df_purchases = df_final.filter(col("Evento") == "purchase")

# 2. Agrupar por dispositivo y contar
device_purchase_ranking = df_purchases.groupBy("Dispositivo") \
    .count() \
    .orderBy("count", ascending=False)

# 3. Mostrar el ranking
print("Ranking de compras por dispositivo:")
device_purchase_ranking.show()

# Obtener el dispositivo líder para el reporte
top_device = device_purchase_ranking.first()
print(f"El dispositivo con mayor número de compras es {top_device['Dispositivo']} con {top_device['count']:,} transacciones.")

Ranking de compras por dispositivo:
+-----------+-----+
|Dispositivo|count|
+-----------+-----+
|    Android|19966|
|        iOS|15069|
|        Web| 9841|
|     Tablet| 2442|
+-----------+-----+

El dispositivo con mayor número de compras es Android con 19,966 transacciones.


### Pregunta 8: Distribución Geográfica
**Calcula el monto total de ventas y el promedio de latencia por Ciudad. ¿Existe alguna ciudad que experimente mayor lentitud (latencia) en sus transacciones?**

Completado el análisis geográfico, se ha identificado que Barranquilla lidera tanto en ventas totales como en latencia promedio.

In [44]:
from pyspark.sql.functions import sum as _sum, avg

# 1. Filtrar registros para el cálculo de latencia realista (sin outliers extremos)
df_latency_clean = df_final.filter(col("Latencia_ms") < 999999)

# 2. Agrupar por ciudad para obtener métricas
geo_analysis = df_final.groupBy("Ciudad").agg(
    _sum("Monto").alias("Ventas_Totales"),
    avg(when(col("Latencia_ms") < 999999, col("Latencia_ms"))).alias("Latencia_Promedio")
).orderBy("Ventas_Totales", ascending=False)

# 3. Mostrar resultados
print("Análisis de Ventas y Latencia por Ciudad:")
geo_analysis.show()

# 4. Identificar ciudad con mayor latencia
slowest_city = geo_analysis.orderBy("Latencia_Promedio", ascending=False).first()
print(f"La ciudad con mayor latencia promedio es {slowest_city['Ciudad']} con {slowest_city['Latencia_Promedio']:.2f} ms.")

Análisis de Ventas y Latencia por Ciudad:
+------------+--------------------+------------------+
|      Ciudad|      Ventas_Totales| Latencia_Promedio|
+------------+--------------------+------------------+
|Barranquilla|1.0923263734999904E8|255.42319499211982|
|        Cali|  9.44676139200002E7|254.65612357252965|
|        NULL| 8.960090003999981E7| 254.7251450567312|
|      Bogotá|  8.44677398200003E7|254.82645034105295|
|   Cartagena| 8.429736819999927E7|  253.951312444408|
|    Medellín| 6.920420601999979E7|254.98137989205918|
+------------+--------------------+------------------+

La ciudad con mayor latencia promedio es Barranquilla con 255.42 ms.


### Pregunta 9: Frecuencia de Usuarios y Conversión

**Identifica los 5 ID_Usuario más activos en la plataforma. ¿Qué porcentaje de sus acciones terminan efectivamente en una compra (purchase)?**

Al filtrar los registros con ID_Usuario NULL, vemos que el usuario identificado más activo es el 37907.0, con 46 acciones y una notable tasa de conversión del 10.87%. Los otros cuatro usuarios en el top 5 mantienen una actividad similar (entre 43 y 44 acciones) pero con tasas de conversión menores, entre el 4.5% y el 7%.

In [47]:
from pyspark.sql.functions import col, count, when

# 1. Filtrar usuarios no nulos y calcular actividad
user_activity = df_final.filter(col("ID_Usuario").isNotNull()) \
    .groupBy("ID_Usuario").agg(
        count("*").alias("Total_Acciones"),
        count(when(col("Evento") == "purchase", True)).alias("Total_Compras")
    )

# 2. Calcular el porcentaje de conversión
user_conversion = user_activity.withColumn(
    "Porcentaje_Conversion",
    (col("Total_Compras") / col("Total_Acciones")) * 100
)

# 3. Obtener los 5 más activos (sin incluir NULLs)
top_5_users = user_conversion.orderBy("Total_Acciones", ascending=False).limit(5)

print("Top 5 Usuarios registrados más activos y su efectividad de compra:")
top_5_users.show()

# Guardar para el reporte técnico
top_user = top_5_users.first()
print(f"El usuario registrado más activo es {top_user['ID_Usuario']} con {top_user['Total_Acciones']} acciones y una tasa de conversión del {top_user['Porcentaje_Conversion']:.2f}%.")

Top 5 Usuarios registrados más activos y su efectividad de compra:
+----------+--------------+-------------+---------------------+
|ID_Usuario|Total_Acciones|Total_Compras|Porcentaje_Conversion|
+----------+--------------+-------------+---------------------+
|   37907.0|            46|            5|   10.869565217391305|
|   29439.0|            44|            2|    4.545454545454546|
|   14954.0|            44|            2|    4.545454545454546|
|   42280.0|            43|            3|    6.976744186046512|
|   33490.0|            43|            2|    4.651162790697675|
+----------+--------------+-------------+---------------------+

El usuario registrado más activo es 37907.0 con 46 acciones y una tasa de conversión del 10.87%.


### Pregunta 10: Análisis Temporal
**Agrupa los datos por hora (extrayendo la hora de la columna Timestamp). ¿En qué franja horaria se detecta el pico más alto de actividad de los usuarios?**

El pico más alto de actividad ocurre a las 4:00 AM con 41,183 registros

In [48]:
from pyspark.sql.functions import hour

# 1. Extraer la hora del timestamp y crear una nueva columna
hourly_traffic = df_final.withColumn("Hora", hour(col("Timestamp")))

# 2. Agrupar por hora y contar el volumen de actividad
traffic_by_hour = hourly_traffic.groupBy("Hora") \
    .count() \
    .orderBy("count", ascending=False)

print("Distribución de actividad por hora (Top 5):")
traffic_by_hour.show(5)

# 3. Identificar la hora pico
peak_hour_row = traffic_by_hour.first()
print(f"El pico más alto de actividad se detecta a las {peak_hour_row['Hora']}:00 horas con {peak_hour_row['count']:,} registros.")

Distribución de actividad por hora (Top 5):
+----+-----+
|Hora|count|
+----+-----+
|   4|41183|
|   7|41126|
|  11|41123|
|   2|41106|
|   9|41098|
+----+-----+
only showing top 5 rows
El pico más alto de actividad se detecta a las 4:00 horas con 41,183 registros.
